# Experiment 7 — External Validation (MIMIC → eICU)

Trains on MIMIC-IV and evaluates on the held-out eICU dataset to test
generalizability across hospital systems.

**Key outputs:**
- AUROC/AUPRC on external eICU cohort
- Performance drop from internal → external
- Calibration shift analysis
- Per-cancer-type generalization table

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.seeding import seed_everything
from src.features.feature_pipeline import CancerTriageFeaturePipeline, load_horizon_data, create_train_val_test_split
from src.models.baselines import BaselineModelTrainer
from src.evaluation.metrics import calculate_clinical_metrics
from src.evaluation.statistical_tests import delong_auc_test

seed_everything(42)
sns.set_theme(style='whitegrid', font_scale=1.2)
os.makedirs('../results/figures', exist_ok=True)
print('Ready.')

In [ ]:
cfg = {
    'seed': 42,
    'paths': {'data_processed': '../data/processed', 'figures': '../results/figures'},
    'features': {'cbc': ['wbc','hemoglobin','platelets','neutrophils','lymphocytes'],
                 'metabolic': ['glucose','albumin','creatinine'],
                 'inflammatory': ['crp']}
}

# Train on MIMIC
X_mimic, y_mimic = load_horizon_data('../data/processed', horizon_months=12, dataset='mimic')
X_train, X_val, _, y_train, y_val, _ = create_train_val_test_split(X_mimic, y_mimic)

pipeline = CancerTriageFeaturePipeline(cfg)
X_train_p = pipeline.fit_transform(X_train, y_train)
X_val_p   = pipeline.transform(X_val)

trainer = BaselineModelTrainer(cfg)
results = trainer.train_all(X_train_p, y_train, X_val_p, y_val)
best = results['xgboost']['model']
print('Model trained on MIMIC-IV.')

In [ ]:
# Evaluate on eICU (external)
X_eicu, y_eicu = load_horizon_data('../data/processed', horizon_months=12, dataset='eicu')
X_eicu_p = pipeline.transform(X_eicu)  # use MIMIC-fitted pipeline

drop_cols = ['subject_id','cancer_type','gender','age']
Xte_mimic_test = pipeline.transform(X_val).drop(columns=[c for c in drop_cols if c in X_val_p.columns], errors='ignore').fillna(0)
Xte_eicu = X_eicu_p.drop(columns=[c for c in drop_cols if c in X_eicu_p.columns], errors='ignore').fillna(0)

# MIMIC internal test
_, _, X_mimic_test, _, _, y_mimic_test = create_train_val_test_split(X_mimic, y_mimic)
X_mimic_test_p = pipeline.transform(X_mimic_test).drop(columns=[c for c in drop_cols if c in X_mimic_test.columns], errors='ignore').fillna(0)

prob_mimic = best.predict_proba(X_mimic_test_p)[:, 1]
prob_eicu  = best.predict_proba(Xte_eicu)[:, 1]

m_mimic = calculate_clinical_metrics(y_mimic_test, prob_mimic)
m_eicu  = calculate_clinical_metrics(y_eicu, prob_eicu)

comparison = pd.DataFrame([m_mimic, m_eicu], index=['MIMIC-IV (internal)', 'eICU (external)'])
print(comparison[['auroc','auprc','f1_score','brier_score']].round(4))

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
comparison[['auroc','auprc']].plot(kind='bar', ax=ax, color=['#2196F3','#FF5722'])
ax.set_title('Internal vs External Validation Performance')
ax.set_ylim(0.5, 1.0)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../results/figures/exp7_external_validation.png', dpi=150, bbox_inches='tight')
plt.show()

comparison.to_csv('../results/exp7_external_validation.csv')
print('Saved.')